<a href="https://colab.research.google.com/github/BSaikiran22/Arrays/blob/main/harassementDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import string
import random
import time
import nltk
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from sklearn import svm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import twitter_samples, stopwords
from nltk.stem.wordnet import WordNetLemmatizer
from nltk import classify, NaiveBayesClassifier

# Download necessary NLTK resources
nltk.download('twitter_samples')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')


[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
import re
import string
import nltk
from nltk.tag import pos_tag
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import stopwords

# Download necessary NLTK resources
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')

def remove_noise(tweet_tokens, stop_words=()):
    cleaned_tokens = []

    # Remove noise using regular expressions (URLs, @mentions, hashtags)
    for token, tag in pos_tag(tweet_tokens):
        token = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+#]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', token)
        token = re.sub(r"(@[A-Za-z0-9_]+)", "", token)

        # Part-of-Speech (POS) tagging for lemmatization
        if tag.startswith("NN"):
            pos = 'n'  # Noun
        elif tag.startswith('VB'):
            pos = 'v'  # Verb
        else:
            pos = 'a'  # Adjective (default)

        lemmatizer = WordNetLemmatizer()
        token = lemmatizer.lemmatize(token, pos)

        # Remove empty tokens, punctuation, and stopwords
        if token and token not in string.punctuation and token.lower() not in stop_words:
            cleaned_tokens.append(token.lower())

    return cleaned_tokens


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
def get_all_words(cleaned_tokens_list):
    for tokens in cleaned_tokens_list:
        for token in tokens:
            yield token

In [5]:
def get_tweets_for_model(cleaned_tokens_list):
    for tweet_tokens in cleaned_tokens_list:
        yield dict([token, True] for token in tweet_tokens)

In [6]:
import nltk
from nltk.corpus import twitter_samples

nltk.download("twitter_samples")

# Load built-in Twitter datasets
positive_tweets = twitter_samples.strings("positive_tweets.json")
negative_tweets = twitter_samples.strings("negative_tweets.json")
all_tweets = twitter_samples.strings("tweets.20150430-223406.json")

print("Loaded:", len(positive_tweets), "positive tweets")
print("Loaded:", len(negative_tweets), "negative tweets")
print("Loaded:", len(all_tweets), "tweets in total")







[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!


Loaded: 5000 positive tweets
Loaded: 5000 negative tweets
Loaded: 20000 tweets in total


# New Section

# New Section

In [7]:
from google.colab import files
uploaded = files.upload()


Saving train.csv to train.csv


In [8]:
import pandas as pd

trainData = pd.read_csv("train.csv")


In [9]:
print(trainData.head())  # Check if the data is loaded properly
trainData = trainData.loc[:, ~trainData.columns.str.startswith('Unnamed:')]
trainData

   Test ID                 Test Scenario                Test Step  \
0        1  Digital harassment Detection  Enter Social Media text   
1        2  Digital harassment Detection  Enter Social Media text   
2        3  Digital harassment Detection  Enter Social Media text   
3        4  Digital harassment Detection  Enter Social Media text   
4        5  Digital harassment Detection  Enter Social Media text   

                                           Test Data  \
0              I hate you.You are so fat @raashi_77!   
1  Hey Guys please folow for more updates.Love yo...   
2                         Wow This is so amazing!!<3   
3  rinamalhotra is so ugly.I don't understand who...   
4  Congrats @SportChampionRio on your yet again a...   

                  Expected Output                   Actual Output Result  \
0       Detect Digital harassment     Digital harassment Detected   PASS   
1  No Digital harassment Detected  No Digital harassment Detected   PASS   
2  No Digital harass

,Test ID,Test Scenario,Test Step,Test Data,Expected Output,Actual Output,Result
0,1,Digital harassment Detection,Enter Social Media text,I hate you.You are so fat @raashi_77!,Detect Digital harassment,Digital harassment Detected,PASS
1,2,Digital harassment Detection,Enter Social Media text,Hey Guys please folow for more updates.Love yo...,No Digital harassment Detected,No Digital harassment Detected,PASS
2,3,Digital harassment Detection,Enter Social Media text,Wow This is so amazing!!<3,No Digital harassment Detected,No Digital harassment Detected,PASS
3,4,Digital harassment Detection,Enter Social Media text,rinamalhotra is so ugly.I don't understand who...,Detect Digital harassment,Digital harassment Detected,PASS
4,5,Digital harassment Detection,Enter Social Media text,Congrats @SportChampionRio on your yet again a...,No Digital harassment Detected,No Digital harassment Detected,PASS
5,6,Digital harassment Detection,Enter Social Media text,is a failure. A mere joke in the name of a sta...,Detect Digital harassment,Digital harassment Detected,PASS
6,7,Digital harassment Detection,Enter Social Media text,OMG !! So damn BEAUTIFUL,No Digital harassment Detected,No Digital harassment Detected,PASS
7,8,Digital harassment Detection,Enter Social Media text,Distinguish Hope they Shoot themselves.#LOL,Detect Digital harassment,Digital harassment Detected,PASS
8,9,Digital harassment Detection,Enter Social Media text,Your song is so annpoing @harsharaj65.Please d...,Detect Digital harassment,Digital harassment Detected,PASS
9,10,Digital harassment Detection,Enter Social Media text,Just Checked in at @hotelgrand Feeling blessed...,No Digital harassment Detected,Digital harassment Detected,PASS


In [10]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv


In [11]:
import pandas as pd

testData = pd.read_csv("test.csv")


In [12]:
testData=testData.loc[:, ~testData.columns.str.startswith('Unnamed:')]
testData

,Test ID,Test Scenario,Test Step,Test Data,Expected Output,Actual Output,Result
0,1,Digital harassment Detection,Enter Social Media text,I hate you.You are so fat @raashi_77!,Detect Digital harassment,Digital harassment Detected,PASS
1,2,Digital harassment Detection,Enter Social Media text,Hey Guys please folow for more updates.Love yo...,No Digital harassment Detected,No Digital harassment Detected,PASS
2,3,Digital harassment Detection,Enter Social Media text,Wow This is so amazing!!<3,No Digital harassment Detected,No Digital harassment Detected,PASS
3,4,Digital harassment Detection,Enter Social Media text,rinamalhotra is so ugly.I don't understand who...,Detect Digital harassment,Digital harassment Detected,PASS
4,5,Digital harassment Detection,Enter Social Media text,Congrats @SportChampionRio on your yet again a...,No Digital harassment Detected,No Digital harassment Detected,PASS
5,6,Digital harassment Detection,Enter Social Media text,is a failure. A mere joke in the name of a sta...,Detect Digital harassment,Digital harassment Detected,PASS
6,7,Digital harassment Detection,Enter Social Media text,OMG !! So damn BEAUTIFUL,No Digital harassment Detected,No Digital harassment Detected,PASS
7,8,Digital harassment Detection,Enter Social Media text,Distinguish Hope they Shoot themselves.#LOL,Detect Digital harassment,Digital harassment Detected,PASS
8,9,Digital harassment Detection,Enter Social Media text,Your song is so annpoing @harsharaj65.Please d...,Detect Digital harassment,Digital harassment Detected,PASS
9,10,Digital harassment Detection,Enter Social Media text,Just Checked in at @hotelgrand Feeling blessed...,No Digital harassment Detected,Digital harassment Detected,PASS


In [13]:
print("-------------SVM Classifier-----------------\n")
vectorizer = TfidfVectorizer(min_df = 5,
                             max_df = 0.95,
                             sublinear_tf = True,
                             use_idf = True)
vectorizer = TfidfVectorizer(min_df=2,
                             max_df=0.8,
                             sublinear_tf=True,
                             use_idf=True)
train_vectors = vectorizer.fit_transform(trainData['Test Data'])
testData['Test Data'] = testData['Test Data'].fillna('')
test_vectors = vectorizer.transform(testData['Test Data'])
classifier_linear = svm.SVC(kernel='linear')
t0 = time.time()
classifier_linear.fit(train_vectors, trainData['Test Data'])
t1 = time.time()
prediction_linear = classifier_linear.predict(test_vectors)
t2 = time.time()
time_linear_train = t1-t0
time_linear_predict = t2-t1
print("Sample Train Data for SVM Classifier:")
print(trainData.sample(frac=1).head(5))
print("\nSVM Classifier Report:-")
print("Training time: %fs; Prediction time: %fs" % (time_linear_train, time_linear_predict))
report = classification_report(testData['Test Data'], prediction_linear, output_dict=True)
print("*NOTE: F1 = 2 * (precision * recall) / (precision + recall)")
print('positive: ', report.get('positive','N/A'))
print('negative: ', report.get('negative','N/A'))

-------------SVM Classifier-----------------

Sample Train Data for SVM Classifier:
   Test ID                 Test Scenario                Test Step  \
4        5  Digital harassment Detection  Enter Social Media text   
8        9  Digital harassment Detection  Enter Social Media text   
3        4  Digital harassment Detection  Enter Social Media text   
9       10  Digital harassment Detection  Enter Social Media text   
0        1  Digital harassment Detection  Enter Social Media text   

                                           Test Data  \
4  Congrats @SportChampionRio on your yet again a...   
8  Your song is so annpoing @harsharaj65.Please d...   
3  rinamalhotra is so ugly.I don't understand who...   
9  Just Checked in at @hotelgrand Feeling blessed...   
0              I hate you.You are so fat @raashi_77!   

                  Expected Output                   Actual Output Result  
4  No Digital harassment Detected  No Digital harassment Detected   PASS  
8       Detect

In [18]:
import nltk
import random
from nltk.corpus import twitter_samples, stopwords
from nltk.classify import NaiveBayesClassifier
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
# Compute accuracy correctly
from nltk.classify import accuracy



# Ensure necessary NLTK resources are downloaded
nltk.download('stopwords')
nltk.download('twitter_samples')
nltk.download('averaged_perceptron_tagger_eng')  # ✅ Correct missing resource
nltk.download('wordnet')
nltk.download('punkt')

# Load stopwords
stop_words = set(stopwords.words('english'))

# Function to clean tokens by removing noise
def remove_noise(tokens, stop_words):
    cleaned_tokens = []
    lemmatizer = WordNetLemmatizer()

    for word, tag in pos_tag(tokens):
        word = word.lower()

        # Remove stopwords & non-alphabetic words
        if word not in stop_words and word.isalpha():
            # Apply lemmatization based on POS tagging
            if tag.startswith("NN"):
                pos = "n"
            elif tag.startswith("VB"):
                pos = "v"
            else:
                pos = "a"

            cleaned_tokens.append(lemmatizer.lemmatize(word, pos))

    return cleaned_tokens

# Load positive & negative tweets from NLTK dataset
positive_tweet_tokens = twitter_samples.tokenized('positive_tweets.json')
negative_tweet_tokens = twitter_samples.tokenized('negative_tweets.json')

# Process tokens to remove noise
positive_cleaned_tokens_list = [remove_noise(tokens, stop_words) for tokens in positive_tweet_tokens]
negative_cleaned_tokens_list = [remove_noise(tokens, stop_words) for tokens in negative_tweet_tokens]

# Convert cleaned tokens into a format suitable for the classifier
def get_tweets_for_model(cleaned_tokens_list):
    return [{word: True for word in tokens} for tokens in cleaned_tokens_list]

positive_tokens_for_model = get_tweets_for_model(positive_cleaned_tokens_list)
negative_tokens_for_model = get_tweets_for_model(negative_cleaned_tokens_list)

# Prepare dataset
positive_dataset = [(tweet_dict, "Positive") for tweet_dict in positive_tokens_for_model]
negative_dataset = [(tweet_dict, "Negative") for tweet_dict in negative_tokens_for_model]

# Merge datasets and shuffle for randomness
dataset = positive_dataset + negative_dataset
random.shuffle(dataset)

# Split dataset into training & testing data
train_data = dataset[:7000]
test_data = dataset[7000:]

# Train Naive Bayes Classifier
classifier = NaiveBayesClassifier.train(train_data)

# Print Classification Report
print("\n-------------NB Classifier-----------------")
print("Naive Bayes Classifier Report:")
print(f"Accuracy: {accuracy(classifier, test_data) * 100:.2f}%")
print("\n**MOST COMMON INFORMATIVE FEATURES:**")
classifier.show_most_informative_features(10)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



-------------NB Classifier-----------------
Naive Bayes Classifier Report:
Accuracy: 72.73%

**MOST COMMON INFORMATIVE FEATURES:**
Most Informative Features
                 welcome = True           Positi : Negati =     32.4 : 1.0
                     sad = True           Negati : Positi =     32.1 : 1.0
                follower = True           Positi : Negati =     21.1 : 1.0
                followed = True           Negati : Positi =     13.4 : 1.0
               community = True           Positi : Negati =     12.4 : 1.0
                   didnt = True           Negati : Positi =     12.3 : 1.0
                 perfect = True           Positi : Negati =     11.7 : 1.0
                    glad = True           Positi : Negati =     11.4 : 1.0
               goodnight = True           Positi : Negati =     11.0 : 1.0
                    miss = True           Negati : Positi =     10.6 : 1.0


In [26]:
#Both Classifiers Are ready for testing with new data sample
print("\n-------------Testing the models-----------------")
print("SAMPLE TEXT1:")
custom_tweet=input()
print(custom_tweet)
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
custom_tokens = remove_noise(word_tokenize(custom_tweet), stop_words)
review_vector = vectorizer.transform([custom_tweet]) # vectorizing
if classifier.classify(dict([token, True] for token in custom_tokens))=='Negative':
        svmresult="Negative"
else:
        svmresult="Positive"
print("SVM Classifier Result:",svmresult)

print("**RESULT**")

if svmresult=='Negative':
            print("Digital harassment is Detected (using SVM Classifier and naive Bayes Classifier)")
else:
            print("Digital harassment is not Detected (using SVM Classifier and naive Bayes Classifier)")



-------------Testing the models-----------------
SAMPLE TEXT1:
Such a bad experience. I regret buying this.
Such a bad experience. I regret buying this.
SVM Classifier Result: Negative
**RESULT**
Digital harassment is Detected (using SVM Classifier and naive Bayes Classifier)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
